# AI Agent Loop Demo

This notebook develops the original starter notebook into a complete, understandable agent-loop exercise.

An **AI agent loop** is a repeated interaction between a language model and tools:
1. The user asks a question.
2. The model decides whether it needs a tool.
3. The application executes the requested Python function.
4. The application sends the tool result back to the model.
5. The model either requests another tool or produces a final answer.

The important point is that the model does **not** directly execute Python. It requests a tool call; our Python program performs it.

---

## What stays faithful to the original notebook?

- Gemini remains the language model.
- `DOCUMENTS` remains the small in-memory knowledge base.
- `search_documents(query)` remains the tool the model can request.
- The agent loop is implemented manually, with automatic function calling disabled.
- Every function call requested in one model turn is executed and returned in one response turn.
- The model's content is appended unchanged so SDK metadata/signatures are preserved.
- The notebook includes follow-up questions, retry handling, and a model connection test.

## What is improved?

- One consistent SDK: `google-genai` (the newer `google.genai` interface).
- One consistent API-key variable, with support for Colab Secrets.
- A valid model setting that can be changed in one place.
- More careful document search, with token matching and ranked results.
- Clear separation between setup, tool definition, model configuration, and the loop.
- Better handling for empty responses, tool errors, missing sources, and rate limits.
- A small test section and extra experiments.

> Run the cells from top to bottom. The notebook is designed for Google Colab, but most cells also work in a local Jupyter notebook.

## 1. Install the Gemini SDK

The original notebook installed `google-generativeai` and later imported `google.genai`. Those are two different SDK interfaces. This version uses the newer **Google Gen AI SDK** consistently.

If Colab reports that a package was already imported, restart the runtime and run the notebook again from the first cell.

In [ ]:
# Install the current Google Gen AI SDK.
# In Colab, run this cell once per fresh runtime.

%pip install -q -U google-genai

print("Google Gen AI SDK installation complete.")

## 2. Load the Gemini API key

Create an API key in Google AI Studio. In Colab, add it to **Secrets** using the name `GOOGLE_API_KEY`, then enable notebook access to that secret.

For a local Jupyter session, set the environment variable before starting the notebook, or enter the key when prompted.

**Never paste a real API key into a notebook cell that you plan to publish or upload to GitHub.**

In [ ]:
import os
import getpass

def load_api_key():
    """
    Load the Gemini API key.

    Priority:
    1. GOOGLE_API_KEY environment variable
    2. Google Colab Secret named GOOGLE_API_KEY
    3. Secure manual prompt
    """
    key = os.environ.get("GOOGLE_API_KEY")

    if key:
        print("API key loaded from environment.")
        return key.strip()

    # Colab Secrets are available only inside Google Colab.
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key.strip()
            print("API key loaded from Colab Secrets.")
            return key.strip()
    except Exception:
        # Not running in Colab, or the secret is not configured.
        pass

    key = getpass.getpass("Paste your Gemini API key: ").strip()
    if not key:
        raise ValueError("No API key was provided.")

    os.environ["GOOGLE_API_KEY"] = key
    print("API key set for this session.")
    return key


API_KEY = load_api_key()

if len(API_KEY) < 15:
    raise ValueError("The API key looks incomplete. Check the copied key.")

## 3. Import the SDK and create the client

The original notebook contained both `google.generativeai` and `google.genai` imports. We use only `google.genai` here.

The model name is kept in one variable. The default below is `gemini-2.5-flash`; if your AI Studio account exposes a different model, change `MODEL_NAME` to an available model that supports `generateContent` and function calling.

In [ ]:
import time
import json
import re
from datetime import datetime

from google import genai
from google.genai import types

MODEL_NAME = "gemini-2.5-flash"

client = genai.Client(api_key=API_KEY)

print("Gemini client created.")
print("Selected model:", MODEL_NAME)

## 4. Create the knowledge base

This section preserves the original notebook's four document topics:
- `ai_basics`
- `machine_learning`
- `ai_applications`
- `ai_ethics`

The documents are intentionally short. The agent can only retrieve information that is present here. Add more documents when you want to experiment with retrieval.

The system instruction later tells Gemini to answer from retrieved material and to say when the documents do not contain an answer.

In [ ]:
DOCUMENTS = {
    "ai_basics": (
        "Artificial intelligence is a branch of computer science that develops "
        "systems capable of performing tasks that normally require human intelligence. "
        "These tasks include learning, reasoning, problem-solving, and understanding language. "
        "AI systems can process large amounts of data to identify patterns and make predictions. "
        "Examples of AI include virtual assistants, recommendation systems, and chatbots."
    ),

    "machine_learning": (
        "Machine learning is a subset of artificial intelligence that enables computers "
        "to learn patterns from data without being explicitly programmed for every task. "
        "Supervised learning uses labelled data to train a model. "
        "Unsupervised learning discovers patterns in data without predefined labels. "
        "Deep learning is a type of machine learning that uses neural networks with multiple layers."
    ),

    "ai_applications": (
        "Artificial intelligence is used in healthcare to help analyze medical images "
        "and support clinical decision-making. "
        "In education, AI tools can provide personalized learning experiences for students. "
        "In agriculture, AI can help monitor crops and detect plant diseases. "
        "In transportation, AI supports route planning, traffic prediction, and driver assistance."
    ),

    "ai_ethics": (
        "AI ethics concerns the responsible design and use of artificial intelligence systems. "
        "AI developers should consider fairness and avoid unjust discrimination. "
        "Personal information should be protected when AI systems collect or process data. "
        "AI decisions should be explainable when they significantly affect people's lives. "
        "Human oversight is important for high-impact decisions made with AI assistance."
    )
}

print("Knowledge base loaded.")
print("Number of documents:", len(DOCUMENTS))
print("Document IDs:", list(DOCUMENTS.keys()))

## 5. Inspect and manage documents

These helper functions make it easier to inspect the knowledge base while experimenting. They do not call Gemini and do not consume API requests.

In [ ]:
def list_documents():
    """Print all document IDs and a short preview."""
    print("DOCUMENT COLLECTION")
    print("-" * 60)

    for name, content in DOCUMENTS.items():
        preview = content[:150]
        if len(content) > 150:
            preview += "..."
        print(f"\n{name}")
        print(preview)


def add_document(name: str, content: str):
    """
    Add a document to the in-memory knowledge base.

    The document is available only while this notebook runtime is active.
    """
    if not name or not name.strip():
        raise ValueError("Document name cannot be empty.")

    if not content or not content.strip():
        raise ValueError("Document content cannot be empty.")

    DOCUMENTS[name.strip()] = content.strip()
    print(f"Added document: {name.strip()}")


def remove_document(name: str):
    """Remove a document by its ID."""
    if name in DOCUMENTS:
        del DOCUMENTS[name]
        print(f"Removed document: {name}")
    else:
        print(f"No document named '{name}' was found.")


list_documents()

## 6. Prepare text for searching

The starter version used `query.lower().split()` and checked whether each query word appeared as a substring in a document. That is easy to understand, but it can produce accidental matches and gives common words too much importance.

This version uses a small transparent tokenizer:
- convert text to lowercase;
- remove punctuation;
- split into words;
- ignore a short list of common words.

This is still keyword search, not semantic search. It is deliberately simple so that the agent loop remains the main learning objective.

In [ ]:
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "can", "do",
    "does", "for", "from", "how", "i", "in", "is", "it", "of",
    "on", "or", "the", "their", "this", "to", "what", "when",
    "where", "which", "why", "with"
}


def tokenize(text: str):
    """Convert a sentence into useful lowercase search terms."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    words = text.split()
    return [word for word in words if word not in STOP_WORDS]


def get_search_terms(query: str):
    """Return unique terms from a search query."""
    return set(tokenize(query))


# Quick tokenizer check.
example_query = "What is the difference between AI and machine learning?"
print("Original:", example_query)
print("Tokens:", tokenize(example_query))

## 7. Define the search tool

This is the Python function that Gemini will be allowed to request.

The function:
1. receives a query string;
2. compares query terms with each document;
3. counts matching terms;
4. sorts documents by match count;
5. returns the top two matching documents;
6. returns a clear message if nothing matches.

The return format includes `[Source: document_id]`, allowing the model to identify where retrieved information came from.

In [ ]:
def search_documents(query: str) -> str:
    """
    Search the document collection for paragraphs relevant to the query.

    Args:
        query: Search phrase supplied by the user or generated by Gemini.

    Returns:
        The top two matching documents, including their source IDs.
    """
    if not isinstance(query, str) or not query.strip():
        return "Please provide a non-empty search query."

    query_words = get_search_terms(query)

    if not query_words:
        return "The query did not contain useful search terms."

    results = []

    for doc_name, content in DOCUMENTS.items():
        document_words = set(tokenize(content))

        # Count how many distinct query terms appear in this document.
        match_count = len(query_words.intersection(document_words))

        if match_count > 0:
            results.append((match_count, doc_name, content))

    # Highest match count first. Use document name as a stable tie-breaker.
    results.sort(key=lambda item: (-item[0], item[1]))

    if not results:
        return "No relevant documents found for that query."

    output = []

    # Return the two most relevant documents, as in the starter notebook.
    for match_count, name, content in results[:2]:
        output.append(
            f"[Source: {name} | Matching terms: {match_count}]\n{content}"
        )

    return "\n\n".join(output)


# Sanity checks: these calls do not use Gemini.
print(search_documents("minimum of 4"))
print("\n" + "=" * 60)
print(search_documents("artificial intelligence definition"))
print("\n" + "=" * 60)
print(search_documents("fairness privacy human oversight"))

## 8. Test the search tool with several queries

Before connecting the tool to Gemini, test it directly. This separates search bugs from model/API issues.

Notice that the final query below is intentionally unrelated to the knowledge base. A good search tool should report that it found no relevant document rather than returning a random paragraph.

In [ ]:
search_examples = [
    "artificial intelligence definition",
    "machine learning supervised learning",
    "AI applications in education and healthcare",
    "fairness privacy and human oversight",
    "how many ghosts eat carrots everyday"
]

for query in search_examples:
    print("\nQUERY:", query)
    print("-" * 60)
    print(search_documents(query))

## 9. Define the tool for Gemini

There are two separate things to understand:

- **Python implementation:** `search_documents()` is the real function that runs in our notebook.
- **Function declaration:** this tells Gemini the tool's name, purpose, and input schema. It does not send the Python source code to Gemini.

The model can request `search_documents` with one required argument named `query`.

In [ ]:
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description=(
                "Search the documents to find the most relevant paragraphs "
                "for a question about artificial intelligence. "
                "Use this tool to retrieve evidence before answering."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": (
                            "A concise search query containing the important "
                            "terms from the user's question."
                        )
                    }
                },
                "required": ["query"]
            }
        )
    ]
)

print("Tool declaration created.")
print("Tool name: search_documents")
print("Required argument: query")

## 10. Configure Gemini

We disable automatic function calling:

```python
automatic_function_calling=types.AutomaticFunctionCallingConfig(
    disable=True
)
```

Why? Because the purpose of this notebook is to see the agent loop. Our code must inspect the model response, run the Python function, and send the result back itself.

The system instruction also sets the evidence rule: use the retrieved documents for questions about the knowledge base, and clearly state when the documents do not contain an answer.

In [ ]:
SYSTEM_INSTRUCTION = """
You are an AI assistant answering questions about Artificial Intelligence.

You have access to a tool named search_documents.

Rules:
1. Use search_documents when you need information from the supplied documents.
2. Base answers about the document collection on the retrieved content.
3. If the documents do not contain the answer, say so clearly.
4. Do not invent a source or claim that a document says something it does not say.
5. Explain answers in clear, student-friendly language.
6. When useful, mention the source document IDs.
7. You may make ordinary conversational replies without searching.
"""

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
    system_instruction=SYSTEM_INSTRUCTION,
    temperature=0.2
)

print("Gemini generation configuration is ready.")

## 11. Check the model connection

This is a small connection test. It is separate from the agent loop and does not use the document-search tool.

If the selected model is not available to your API key, change `MODEL_NAME` in Section 3 to a model shown by the model-listing cell later in this notebook.

In [ ]:
def test_gemini_connection():
    """Send a tiny request to verify the API key and model."""
    print("Testing Gemini connection...", flush=True)

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents="Reply with only the word Hello.",
            config=types.GenerateContentConfig(
                max_output_tokens=10,
                http_options=types.HttpOptions(timeout=30000)
            )
        )

        print("Success:", response.text)
        return True

    except Exception as error:
        print("Connection test failed.")
        print("Error type:", type(error).__name__)
        print("Details:", str(error))
        return False


connection_ok = test_gemini_connection()

if not connection_ok:
    print(
        "\nCheck your API key, model name, internet connection, "
        "and AI Studio project access."
    )

## 12. Add retry handling

Free-tier APIs can temporarily return rate-limit or service errors. The original notebook retried errors containing `429` and waited longer after each attempt.

This version keeps the same basic approach, while making the number of attempts and wait duration configurable. It does not retry every possible error: invalid credentials and invalid model names should be fixed rather than repeatedly retried.

In [ ]:
MAX_RETRIES = 3
RETRY_WAIT_SECONDS = 20


def send_with_retry(contents, max_retries=MAX_RETRIES):
    """
    Send a request to Gemini, retrying temporary rate-limit errors.

    Args:
        contents: The conversation contents to send.
        max_retries: Maximum number of attempts.

    Returns:
        The Gemini response object.
    """
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )

        except Exception as error:
            error_text = str(error)

            is_rate_limit = (
                "429" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
                or "rate limit" in error_text.lower()
            )

            if is_rate_limit and attempt < max_retries - 1:
                wait = RETRY_WAIT_SECONDS * (attempt + 1)
                print(
                    f"Rate limit hit. Waiting {wait} seconds "
                    f"(attempt {attempt + 1}/{max_retries})..."
                )
                time.sleep(wait)
            else:
                raise

    raise RuntimeError("Gemini request failed after all retry attempts.")

## 13. Understand the manual agent loop

This is the central part of the notebook.

A model response can contain:
- ordinary text, meaning the model is ready to answer; or
- one or more function calls, meaning the model wants our program to run a tool.

The loop below:
1. sends the current conversation to Gemini;
2. obtains the model's response content;
3. appends that content **unchanged** to the conversation;
4. checks for function calls;
5. executes each requested function in Python;
6. returns all tool results in one user message;
7. repeats until the model gives a normal text answer or the step limit is reached.

Appending the model content unchanged is important for preserving SDK metadata, including thought signatures when the model/API uses them.

In [ ]:
def run_agent(question: str, max_steps: int = 8) -> str:
    """
    Run the manual Gemini agent loop for one question.

    Args:
        question: The user's question.
        max_steps: Maximum number of model turns allowed.

    Returns:
        The final answer produced by Gemini.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question cannot be empty.")

    print(f"Question: {question}\n" + "-" * 60)

    # Start a fresh conversation for this question, matching the original
    # notebook's run_agent behavior.
    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)

        # A normal response should contain at least one candidate.
        if not response.candidates:
            print("The model returned no response candidates.")
            return "(Stopped: empty model response)"

        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED.
        # Do not rebuild this content from text alone.
        # This preserves metadata/signatures required by some models.
        contents.append(model_content)

        # Collect every function call in this model turn.
        calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call is not None
        ]

        # No function call means the model has produced its final answer.
        if not calls:
            final_text = response.text or ""
            print(f"\nFinal answer:\n{final_text}")
            return final_text

        # Run every requested tool call and return all results in ONE turn.
        result_parts = []

        for call in calls:
            args = dict(call.args or {})

            print(
                f"Step {step}: model calls "
                f"{call.name}({args})"
            )

            # Route the requested function to our Python implementation.
            if call.name == "search_documents":
                try:
                    result = search_documents(**args)
                except Exception as error:
                    result = (
                        f"Tool error: {type(error).__name__}: {error}"
                    )
            else:
                # Defensive handling in case the model requests an
                # undeclared or unexpected tool name.
                result = f"Unknown tool: {call.name}"

            print(f"  Result preview: {str(result)[:160]}...")

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        # All function results are sent together in one user turn.
        contents.append(
            types.Content(
                role="user",
                parts=result_parts
            )
        )

    print("\nAgent stopped: maximum number of steps reached.")
    return "(Stopped: hit max_steps)"

## 14. Run the first agent question

This question should cause Gemini to request `search_documents`, receive the `ai_basics` material, and then write an answer.

Watch the printed output:
- `Question` is the original user input.
- `Step 1` shows the tool request made by Gemini.
- `Result preview` shows the Python tool's response.
- `Final answer` is the next model response after it receives the tool result.

In [ ]:
first_answer = run_agent("What is artificial intelligence?")
print("\nFirst run completed.")

## 15. Inspect the complete agent sequence

The next cell prints the same logical sequence as a compact diagram. It does not make another API request.

In [ ]:
agent_loop_steps = [
    "User enters a question",
    "Question is placed in conversation contents",
    "Gemini receives the conversation",
    "Gemini returns a function call",
    "Python executes search_documents(query)",
    "Python wraps the result as a function response",
    "The result is appended to conversation contents",
    "Gemini receives the tool result",
    "Gemini returns another tool call or a final answer"
]

for number, description in enumerate(agent_loop_steps, start=1):
    print(f"{number:02d}. {description}")

## 16. Ask follow-up questions

These questions preserve the original notebook's learning goals:
- compare AI and machine learning;
- ask something absent from the database;
- ask about AI ethics;
- ask about applications in education and healthcare;
- ask why fairness, privacy, and human oversight matter.

The notebook intentionally starts a fresh `run_agent` conversation for each question, just like the original implementation. The model can make multiple tool calls within that one question, but these separate questions do not share chat history.

The delay helps avoid sending requests too quickly.

In [ ]:
questions = [
    "What is the difference between artificial intelligence and machine learning?",

    # This question is intentionally not answered by the database.
    "How many ghosts eat carrots every day?",

    "What are the key ethical concerns when developing AI systems?",

    "What are the main applications of artificial intelligence in education and healthcare?",

    "Why are fairness, privacy, and human oversight important in AI ethics?"
]

print("Number of follow-up questions:", len(questions))

In [ ]:
def run_question_batch(question_list, delay_seconds=15):
    """
    Run multiple independent questions.

    A delay is inserted between questions to reduce the chance of hitting
    API rate limits. Each question gets its own fresh agent conversation.
    """
    for index, question in enumerate(question_list):
        if index > 0:
            print(
                f"\nWaiting {delay_seconds} seconds "
                "before the next request..."
            )
            time.sleep(delay_seconds)

        print("\n" + "=" * 60)
        print(f"QUESTION {index + 1} OF {len(question_list)}")
        print("=" * 60)

        run_agent(question)


# Run the batch when you are ready.
# Uncomment the next line to execute all five questions.
# run_question_batch(questions, delay_seconds=15)

## 17. Ask your own question

Use this cell to enter a question interactively. The question should preferably be about one of the four document topics.

Try asking:
- "Explain supervised and unsupervised learning."
- "What AI applications are mentioned in agriculture?"
- "What does the document say about fairness?"
- "Does the database explain how neural networks work?"

The last question is useful because the database only briefly mentions neural networks. The model should avoid pretending that the documents contain a detailed explanation.

In [ ]:
user_question = input("Enter your question: ").strip()

if user_question:
    custom_answer = run_agent(user_question)
else:
    print("No question entered. Run this cell again when ready.")

## 18. Add another document and search it

This experiment demonstrates that the tool searches the Python dictionary at runtime. You do not need to redeclare the Gemini tool when you add another document, because the tool's input schema has not changed.

In [ ]:
add_document(
    "robotics",
    (
        "Robotics combines mechanical engineering, electronics, and computer "
        "science to build machines that perform physical tasks. Artificial "
        "intelligence can help robots perceive their surroundings, plan actions, "
        "recognize objects, and adapt to changing conditions. Robots are used "
        "in manufacturing, logistics, research, and some healthcare settings."
    )
)

print(search_documents("How can AI help robots recognize objects?"))

## 19. Ask Gemini about the new document

The next request should retrieve the `robotics` source if the model chooses the search tool.

In [ ]:
robotics_answer = run_agent(
    "How can artificial intelligence help robots recognize objects and plan actions?"
)

## 20. Test a question that is outside the knowledge base

The system instruction asks Gemini to be transparent when the documents do not contain an answer. The tool may be called more than once with different search terms; that is part of the agent's iterative behavior.

Do not judge the search system only by whether it finds something. A correct "no relevant documents" result is also useful.

In [ ]:
missing_information_question = (
    "What is the most popular food item in Canada?"
)

missing_information_answer = run_agent(missing_information_question)

## 21. List models available to your API key

This cell asks the API which models are available. It is useful when the model name configured near the top of the notebook is unavailable.

The returned list can vary by account, region, API access, and date. Select a model that supports content generation and function calling.

In [ ]:
def list_available_models():
    """Print models that support generateContent."""
    print("Models available for content generation")
    print("-" * 60)

    try:
        for model_info in client.models.list():
            supported_actions = model_info.supported_actions or []

            if "generateContent" in supported_actions:
                print(model_info.name)

    except Exception as error:
        print(type(error).__name__, str(error))


list_available_models()

## 22. A more detailed connection test

This repeats the connection test with a longer timeout. It is useful if the first test failed because of a slow network connection.

A timeout is not the same as an invalid API key. Read the error message before changing credentials.

In [ ]:
def detailed_connection_test(timeout_ms=60000):
    """Test Gemini with an explicitly configured HTTP timeout."""
    print("Testing Gemini...", flush=True)

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents="Say Hello.",
            config=types.GenerateContentConfig(
                max_output_tokens=10,
                http_options=types.HttpOptions(timeout=timeout_ms)
            )
        )

        print("Success:", response.text)
        return True

    except Exception as error:
        print("Error:", type(error).__name__, str(error))
        return False


# Run only if you need another connection check.
# detailed_connection_test()

## 23. Basic tests for the local search function

These tests do not contact Gemini and therefore do not consume API requests.

They check:
- the search function returns the expected source for a clear query;
- an unrelated query produces the no-results message;
- an empty query is handled safely;
- adding a document makes it searchable.

In [ ]:
def test_search_finds_ai_basics():
    result = search_documents("artificial intelligence definition")
    assert "[Source: ai_basics" in result


def test_search_finds_machine_learning():
    result = search_documents("supervised learning labelled data")
    assert "[Source: machine_learning" in result


def test_search_returns_no_match():
    result = search_documents("spaceship banana volcano")
    assert "No relevant documents" in result


def test_search_handles_empty_query():
    result = search_documents("")
    assert "non-empty" in result


def test_added_document_is_searchable():
    add_document(
        "temporary_test",
        "Quantum computers use quantum bits called qubits."
    )
    result = search_documents("quantum computers qubits")
    assert "[Source: temporary_test" in result
    remove_document("temporary_test")


def run_local_tests():
    tests = [
        test_search_finds_ai_basics,
        test_search_finds_machine_learning,
        test_search_returns_no_match,
        test_search_handles_empty_query,
        test_added_document_is_searchable
    ]

    failures = []

    for test in tests:
        try:
            test()
            print("[PASS]", test.__name__)
        except Exception as error:
            failures.append((test.__name__, str(error)))
            print("[FAIL]", test.__name__, "-", error)

    print("\n" + "-" * 60)
    print(f"{len(tests) - len(failures)}/{len(tests)} tests passed.")

    if failures:
        print("Failures:", failures)

    return len(failures) == 0


tests_passed = run_local_tests()

## 24. Save a simple execution record

This helper records the question, final answer, timestamp, and selected model. It does not capture private API credentials.

It is a basic demonstration of logging. A production agent would usually record structured tool-call events, timings, errors, and token usage as well.

In [ ]:
def save_answer_record(question, answer, filename="agent_run_record.json"):
    """Save one question and answer to a JSON file."""
    record = {
        "timestamp": datetime.now().isoformat(),
        "model": MODEL_NAME,
        "question": question,
        "answer": answer
    }

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(record, file, indent=2, ensure_ascii=False)

    print("Saved:", filename)


# Example:
# save_answer_record(
#     "What is artificial intelligence?",
#     first_answer
# )

## 25. Troubleshooting

### `ModuleNotFoundError: No module named 'google.genai'`
Run the installation cell. If the package was installed after imports had already run, restart the runtime and execute cells from the beginning.

### `API key not found` or authentication error
Check that the Colab Secret is named exactly `GOOGLE_API_KEY`, or set the environment variable in your local session. Do not print the key to debug it.

### `404 NOT_FOUND` or model not found
Change `MODEL_NAME` to a model listed by Section 21. Model availability can differ between accounts.

### `429 RESOURCE_EXHAUSTED`
The API is rate-limiting requests. Wait before retrying and increase the delay between follow-up questions. Your account's actual quotas are shown in AI Studio; do not assume a fixed quota from an old notebook comment.

### The model does not call the search tool
Check that the tool declaration is passed in `config`, that automatic function calling is disabled, and that the question clearly needs information from the documents.

### The model gives an answer when no document matches
The system instruction asks the model to say when the database has no answer. For stricter grounding, add application-side checks or require the model to cite retrieved source IDs. A prompt alone is not a perfect enforcement mechanism.

## 26. Experiments to try

Complete these experiments after the notebook runs successfully.

1. Add a document about renewable energy and ask two questions about it.
2. Change the top-two retrieval limit in `search_documents` to top one.
3. Ask a question whose answer requires two documents.
4. Ask an unrelated question and inspect how many times Gemini calls the tool.
5. Change `max_steps` from 8 to 1 and observe the stopping behavior.
6. Remove the `ai_ethics` document and ask an ethics question.
7. Change the system instruction and compare the answers.
8. Try a different available Gemini model.
9. Print the arguments requested by Gemini before executing the tool.
10. Replace keyword matching with embeddings and compare retrieval quality.

### Questions to check your understanding

- Why does the model need a tool declaration?
- What is the difference between the function declaration and the Python function?
- Why is automatic function calling disabled in this notebook?
- What does the model send back when it wants a tool to run?
- Why do we send tool results back to the model?
- Why are all results from one model turn placed in one user message?
- Why do we append the model's content unchanged?
- What happens if the model keeps requesting tools?
- Why is a maximum step count necessary?
- Why should the application not trust arbitrary tool names or arguments?

## 27. Final summary

The central agent loop can be reduced to this pattern:

```python
for step in range(max_steps):
    response = send_to_model(contents)
    model_content = response.candidates[0].content
    contents.append(model_content)

    calls = get_function_calls(model_content)

    if not calls:
        return response.text

    results = execute_requested_tools(calls)
    contents.append(tool_results_as_one_message(results))
```

The full notebook adds the supporting pieces: API setup, a knowledge base, a search tool, a function declaration, retry handling, tests, follow-up questions, and troubleshooting.

**Core idea:** Gemini chooses *when* to request a tool and *what arguments* to send. Python executes the tool. Gemini receives the result and decides what to do next. Repeating those steps is the agent loop.

In [ ]:
# QUICK REFERENCE
# ---------------
# Search the local knowledge base:
# print(search_documents("AI ethics fairness privacy"))
#
# Add a document:
# add_document("new_topic", "Your paragraph goes here.")
#
# Remove a document:
# remove_document("new_topic")
#
# Ask Gemini:
# run_agent("What is machine learning?")
#
# Run several questions:
# run_question_batch(questions, delay_seconds=15)
#
# List available models:
# list_available_models()
#
# Run local tests:
# run_local_tests()
#
# The notebook uses a fresh conversation for every run_agent() call.
# To implement multi-turn chat history, keep the `contents` list outside
# run_agent() and append new user/model/tool turns to that same list.